In [45]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization, Dense, Dropout, Input
from sklearn.metrics import classification_report, confusion_matrix

In [24]:

df_train = pd.read_csv('train.csv')


df_train = df_train.dropna(subset=['HomePlanet']).copy()


y = df_train['HomePlanet']


columnas_a_quitar = ['HomePlanet', 'PassengerId', 'Name', 'Cabin']
X = df_train.drop(columns=[c for c in columnas_a_quitar if c in df_train.columns])


for col in X.select_dtypes(include=['float64', 'int64']).columns:
    X[col] = X[col].fillna(X[col].median())

for col in X.select_dtypes(include=['string', 'bool']).columns:
    X[col] = X[col].fillna(X[col].mode()[0])

X = pd.get_dummies(X, drop_first=True)


In [25]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)  

X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

In [34]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [41]:
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),  
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax'),
])


In [42]:

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  
    metrics=['accuracy'],
)

In [43]:

early_stop = EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)


In [44]:

history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1,
)

Epoch 1/100
213/213 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6567 - loss: 0.7783 - val_accuracy: 0.7192 - val_loss: 0.6813
Epoch 2/100
213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 897us/step - accuracy: 0.6998 - loss: 0.6848 - val_accuracy: 0.7416 - val_loss: 0.6014
Epoch 3/100
213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 969us/step - accuracy: 0.7126 - loss: 0.6513 - val_accuracy: 0.7422 - val_loss: 0.5809
Epoch 4/100
213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 875us/step - accuracy: 0.7222 - loss: 0.6342 - val_accuracy: 0.7475 - val_loss: 0.5734
Epoch 5/100
213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 998us/step - accuracy: 0.7262 - loss: 0.6198 - val_accuracy: 0.7440 - val_loss: 0.5660
Epoch 6/100
213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7322 - loss: 0.6078 - val_accuracy: 0.7534 - val_loss: 0.5615
Epoch 7/100
213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7352 - loss: 0.6045 - val_accuracy: 0.7528 - val_loss: 0.5515
Epoch 8/100
213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 938us/step - accuracy: 0.7403 - loss: 0.5944 

In [46]:

val_loss, val_acc = model.evaluate(X_val_scaled, y_val, verbose=0)
print(f"Pérdida en validación (val_loss): {val_loss:.4f}")
print(f"Precisión en validación (val_acc): {val_acc:.4f}\n")


probabilidades = model.predict(X_val_scaled)
y_pred = np.argmax(probabilidades, axis=1)


print("Reporte de Clasificación:")
print(classification_report(y_val, y_pred, target_names=le.classes_))


print("Matriz de Confusión:")
print(confusion_matrix(y_val, y_pred))


Pérdida en validación (val_loss): 0.5078
Precisión en validación (val_acc): 0.7646

54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 924us/step
Reporte de Clasificación:
              precision    recall  f1-score   support

       Earth       0.85      0.82      0.84       921
      Europa       0.65      0.88      0.75       426
        Mars       0.72      0.47      0.57       352

    accuracy                           0.76      1699
   macro avg       0.74      0.73      0.72      1699
weighted avg       0.77      0.76      0.76      1699

Matriz de Confusión:
[[756 114  51]
 [ 37 377  12]
 [ 96  90 166]]
